# Notebook 02 – Areal-Interpolation & Datenvorbereitung (ECHTE Daten)
### Harmonisierung der LOR-Geometrien 2013-2019 (447 PLR) auf 2021 (542 PLR)

Ablauf:
1. Beide Geodaten-Versionen laden
2. Raeumliche Verschneidung + flaechengewichtete Interpolation
3. MSS-Werte 2013-2019 auf neue Geometrie uebertragen
4. Mit 2021/2023 (bereits neue Geometrie) zusammenfuehren
5. Zusatzfeatures (OePNV, Gruen, Laerm) berechnen
6. Split, Normalisierung, Speichern -> bereit fuer Fine-Tuning

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 1: Imports
# ═══════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import pickle
import os
import requests
import tempfile
import warnings
warnings.filterwarnings('ignore')

from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

for folder in ['../data/raw', '../data/processed', '../data/final',
               '../models', '../results']:
    os.makedirs(folder, exist_ok=True)

print('✅ Setup abgeschlossen!')
print('''
BENOETIGTE DATEIEN (bitte vorher in data/raw/ ablegen):
  data/raw/mss_2013.xlsx ... mss_2023.xlsx  (Index-Indikatoren)

  Geometrien sind OPTIONAL lokal ablegbar - sonst automatischer
  Online-Fallback in Zelle 3 (kein manueller Download noetig):
  data/raw/planungsraeume_alt/*.shp   (LOR-Geometrie bis 2019, 447 PLR)
  data/raw/planungsraeume_neu/*.shp   (LOR-Geometrie ab 2021, 542 PLR)
''')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 2: MSS-Loader-Funktion (validierte Struktur, alle Jahre)
# ═══════════════════════════════════════════════════════════════
# Struktur wurde an den echten Dateien geprueft (2013-2025):
#  - 2013 hat 2 Sheets, das zweite ('...IndexInd_Ant_PLR_MSS...') ist das Datenblatt
#  - Alle Jahre: Zeile 8 (0-indiziert) = Spaltenkopf, Zeile 9-10 = MW/SD, Zeile 11 = leer
#  - Spalten: Nummer, Name, EW, S1, S2, S3, S4, D1, D2, D3, D4, (Hinweis)
#  - S2 wechselt Bedeutung: 2013-2017 Langzeit-AL, 2019+2021 fehlt, 2023+2025 Alleinerziehende
#    -> S2 wird bewusst NICHT verwendet (siehe Kapitel 3.2 / 6.3 der Arbeit)
#  - 'nicht beruecksichtigte' PLR (EW<300, Ausreisser) haben 0-Werte + Hinweistext -> rausfiltern

def load_mss_year(path, year, sheet_name=None):
    """Laedt eine MSS Index-Indikatoren Excel-Datei (Struktur 2013-2025)."""

    xl = pd.ExcelFile(path)

    # 2013 hat 2 Sheets - das Datenblatt ist NICHT 'Erlaeuterungen'
    if sheet_name is None:
        data_sheets = [s for s in xl.sheet_names if 'Erläuterungen' not in s and 'Erlaeuterungen' not in s]
        sheet_name = data_sheets[0] if data_sheets else xl.sheet_names[0]

    df = pd.read_excel(xl, sheet_name=sheet_name, header=8, skiprows=[9, 10, 11])

    # Erste 3 Spalten sind immer: Nummer, Name, EW - Rest nach Position benennen
    # (robuster als nach Namen, da EW-Spaltenname jaehrlich leicht variiert)
    cols = list(df.columns)
    rename_map = {
        cols[0]: 'plr_id_alt',
        cols[1]: 'plr_name',
        cols[2]: 'einwohner',
    }
    # S1, S2, S3, S4 stehen an Position 3,4,5,6 (falls S2 fehlt: nur 3 Status-Spalten!)
    # Wir identifizieren sie ueber die Spaltennamen 'S1'..'S4', nicht ueber Position,
    # da 2019/2021 keine 'S2'-Spalte haben und sich dadurch alles verschiebt.
    for c in cols:
        cs = str(c).strip()
        if cs == 'S1':
            rename_map[c] = 'arbeitslosigkeit'
        elif cs == 'S3':
            rename_map[c] = 'sgb2_quote'
        elif cs == 'S4':
            rename_map[c] = 'kinderarmut'

    df = df.rename(columns=rename_map)

    keep = ['plr_id_alt', 'plr_name', 'einwohner',
            'arbeitslosigkeit', 'sgb2_quote', 'kinderarmut']
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()

    # 'Nicht beruecksichtigte' PLR rausfiltern:
    # -> plr_id_alt muss eine echte PLR-Nummer sein (8 Ziffern als String/Zahl)
    df = df.dropna(subset=['plr_id_alt'])
    df['plr_id_alt'] = df['plr_id_alt'].astype(str).str.strip()
    df = df[df['plr_id_alt'].str.match(r'^\d{6,8}$')]

    # Zeilen mit EW=0 UND allen Werten=0 sind Platzhalter (< 300 EW / Ausreisser) -> raus
    value_cols = [c for c in ['arbeitslosigkeit','sgb2_quote','kinderarmut'] if c in df.columns]
    all_zero = (df[value_cols] == 0).all(axis=1)
    df = df[~all_zero]

    df['year'] = year
    return df.reset_index(drop=True)


print('✅ Loader-Funktion definiert!')
print('\nS2-Hinweis: bewusst NICHT geladen (siehe Notiz oben) -')
print('Bedeutung wechselt zwischen Berichtsjahren, keine konsistente Zeitreihe moeglich.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 3: Geodaten laden (alte + neue LOR)
# ═══════════════════════════════════════════════════════════════
# Ladereihenfolge je Geometrie: 1) lokale Datei falls vorhanden,
# 2) sonst Online-Quelle (funktioniert auch wenn fbinter offline ist).
#
# WICHTIG: GitHub raw-Links fuer .geojson in diesem Repo sind
# Git-LFS-Zeiger (liefern nur Text-Pointer, keine echten Daten) -
# deshalb wird stattdessen der Technologiestiftung-Berlin S3-Bucket
# genutzt, der echte KML-Dateien mit Polygon-Geometrie bereitstellt.

def load_geometry(local_path, online_url, name):
    """Laedt Geometrie lokal falls vorhanden, sonst per Download von online_url."""
    if os.path.exists(local_path):
        print(f'📁 {name}: lokale Datei gefunden ({local_path})')
        return gpd.read_file(local_path)

    print(f'🌐 {name}: keine lokale Datei - lade online...')
    response = requests.get(online_url, timeout=60)
    response.raise_for_status()

    suffix = '.kml' if online_url.endswith('.kml') else '.geojson'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False, mode='wb') as tmp:
        tmp.write(response.content)
        tmp_path = tmp.name

    try:
        gdf = gpd.read_file(tmp_path)
    finally:
        os.remove(tmp_path)

    print(f'✅ {name}: online geladen ({len(gdf)} Features)')
    return gdf


# --- Alte LOR (2013-2019, 447 PLR) ---
# Quelle: Technologiestiftung Berlin Open Data (S3), KML mit
# echten Polygonen. PLR-ID steckt im Feld 'broker Dow'.
ALT_LOKAL = '../data/raw/planungsraeume_alt/planungsraeume.shp'
ALT_ONLINE = (
    'https://tsb-opendata.s3.eu-central-1.amazonaws.com/'
    'lor_planungsgraeume/lor_planungsraeume.kml'
)

gdf_alt = load_geometry(ALT_LOKAL, ALT_ONLINE, 'Alte LOR')
gdf_alt = gdf_alt.set_crs(epsg=4326, allow_override=True).to_crs(epsg=25833)

print(f'   Spalten: {gdf_alt.columns.tolist()}')

# PLR-ID Spalte automatisch erkennen - WICHTIG: 'DOW'/'broker Dow' hat
# PRIORITAET vor 'PLANUNGSRA', da letzteres in der KML-Quelle den PLR-
# NAMEN (z.B. 'Stuelerstrasse') enthaelt, NICHT die numerische ID!
# Verifiziert gegen die offizielle CSV-Version derselben Quelle:
# Spalten BEZIRKSNAM,PLANUNGSRA,DATUM_GUEL,FLAECHENGR,'broker Dow' -
# 'broker Dow' enthaelt die echte 8-stellige ID (z.B. "01011101").
id_col_alt = next(
    (c for c in gdf_alt.columns if 'DOW' in c.upper()),
    next(
        (c for c in gdf_alt.columns if c.upper() == 'PLR_ID'),
        gdf_alt.columns[0]
    )
)
print(f'   -> Als plr_id_alt erkannt: Spalte "{id_col_alt}"')
print(f'   -> Beispielwerte: {gdf_alt[id_col_alt].head(3).tolist()}')

gdf_alt = gdf_alt.rename(columns={id_col_alt: 'plr_id_alt'})
gdf_alt = gdf_alt[['plr_id_alt', 'geometry']]
gdf_alt['plr_id_alt'] = gdf_alt['plr_id_alt'].astype(str).str.strip().str.zfill(8)
gdf_alt['flaeche_alt'] = gdf_alt.geometry.area

print(f'   -> Nach zfill(8): {gdf_alt["plr_id_alt"].head(3).tolist()}')


# --- Neue LOR (2021-2023, 542 PLR) ---
# Quelle: amtlicher WFS-Dienst (Amt fuer Statistik Berlin-Brandenburg,
# Stand 01/2023) - funktioniert auch bei fbinter-Ausfall (anderer Server)
NEU_LOKAL = '../data/raw/planungsraeume_neu/LOR_2023-01-01_PLR_EPSG_25833_nur_ID.shp'

# Mehrere Formate/Endpunkt-Varianten probieren (GeoServer-Konfigurationen variieren)
NEU_ONLINE_VARIANTEN = [
    'https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=2.0.0&request=GetFeature&typeNames=lor_2021:a_lor_plr_2021&outputFormat=application/json',
    'https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=2.0.0&request=GetFeature&typeNames=lor_2021:a_lor_plr_2021&outputFormat=json',
    'https://gdi.berlin.de/services/wfs/lor_2021?service=WFS&version=1.1.0&request=GetFeature&typeName=lor_2021:a_lor_plr_2021&srsName=EPSG:25833',
    'https://opendata.arcgis.com/datasets/9d6a9bd32d0f480698f5cee208b27f90_0.geojson',
]

gdf_neu = None
if os.path.exists(NEU_LOKAL):
    print(f'📁 Neue LOR: lokale Datei gefunden ({NEU_LOKAL})')
    gdf_neu = gpd.read_file(NEU_LOKAL)
else:
    for versuch_nr, url in enumerate(NEU_ONLINE_VARIANTEN, 1):
        try:
            print(f'🌐 Neue LOR: Versuch {versuch_nr}/{len(NEU_ONLINE_VARIANTEN)}...')
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            suffix = '.geojson' if 'json' in url else '.gml'
            with tempfile.NamedTemporaryFile(suffix=suffix, delete=False, mode='wb') as tmp:
                tmp.write(response.content)
                tmp_path = tmp.name
            try:
                gdf_neu = gpd.read_file(tmp_path)
                print(f'✅ Neue LOR: online geladen ({len(gdf_neu)} Features, Versuch {versuch_nr})')
                os.remove(tmp_path)
                break
            except Exception as e_read:
                os.remove(tmp_path)
                print(f'   ⚠️ Versuch {versuch_nr} nicht lesbar: {e_read}')
                continue
        except Exception as e_req:
            print(f'   ⚠️ Versuch {versuch_nr} fehlgeschlagen: {e_req}')
            continue

if gdf_neu is None:
    raise RuntimeError(
        'Neue LOR (542 PLR) konnte weder lokal noch online geladen werden. '
        'Bitte manuell von https://gdi.berlin.de/viewer/main/ herunterladen '
        'und in data/raw/planungsraeume_neu/ ablegen.'
    )

gdf_neu = gdf_neu.to_crs(epsg=25833)

print(f'   Spalten: {gdf_neu.columns.tolist()}')
id_col_neu = next(
    (c for c in gdf_neu.columns if c.upper() in ('PLR_ID', 'PLR_ID_STR', 'SCHLUESSEL')),
    gdf_neu.columns[0]
)
print(f'   -> Als plr_id erkannt: Spalte "{id_col_neu}"')

gdf_neu = gdf_neu.rename(columns={id_col_neu: 'plr_id'})
gdf_neu = gdf_neu[['plr_id', 'geometry']]
gdf_neu['plr_id'] = gdf_neu['plr_id'].astype(str).str.strip().str.zfill(8)
gdf_neu['flaeche_neu'] = gdf_neu.geometry.area

print(f'\n✅ Alte LOR:  {len(gdf_alt)} Planungsraeume (erwartet: 447)')
print(f'✅ Neue LOR:  {len(gdf_neu)} Planungsraeume (erwartet: 542)')

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
gdf_alt.plot(ax=axes[0], color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title(f'Alte LOR ({len(gdf_alt)} PLR)')
axes[0].set_axis_off()
gdf_neu.plot(ax=axes[1], color='orange', edgecolor='white', linewidth=0.3)
axes[1].set_title(f'Neue LOR ({len(gdf_neu)} PLR)')
axes[1].set_axis_off()
plt.tight_layout()
plt.savefig('../results/02_lor_geometrien.png', dpi=150)
plt.show()

# ── Geometrie fuer Notebook 01 (Pre-Training) cachen ────────
geo_cache = gdf_neu[['plr_id']].copy()
geo_cache['centroid_x'] = gdf_neu.geometry.centroid.x
geo_cache['centroid_y'] = gdf_neu.geometry.centroid.y
alex_pt = Point(392456, 5820118)
geo_cache['dist_zentrum'] = (gdf_neu.geometry.centroid.distance(alex_pt) / 1000).round(2)
geo_cache.to_csv('../data/processed/plr_geometrie_cache.csv', index=False)
print(f'✅ Geometrie gecacht fuer Notebook 01: {len(geo_cache)} PLR '
      f'-> data/processed/plr_geometrie_cache.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 4: Raeumliche Verschneidung (Areal-Interpolation Kernstueck)
# ═══════════════════════════════════════════════════════════════

print('🔄 Berechne raeumliche Verschneidung (kann 1-3 Min dauern)...')

overlay = gpd.overlay(gdf_alt, gdf_neu, how='intersection', keep_geom_type=True)
overlay['ueberlappung_flaeche'] = overlay.geometry.area

# Gewicht = Anteil der Ueberlappung an der NEUEN (Ziel-)Flaeche
overlay['gewicht'] = overlay['ueberlappung_flaeche'] / overlay['flaeche_neu']

print(f'✅ Verschneidung: {len(overlay)} Teilflaechen')

# Qualitaetskontrolle: Gewichte je neuem PLR sollten zu ~1 summieren
check = overlay.groupby('plr_id')['gewicht'].sum()
print(f'\nQualitaetskontrolle (Summe Gewichte je neuem PLR, sollte ~1.0 sein):')
print(check.describe().round(3))

problematisch = check[(check < 0.9) | (check > 1.1)]
if len(problematisch) > 0:
    print(f'\n⚠️  {len(problematisch)} PLR mit Abweichung >10% (Randeffekte/Gewaesser) - normal in geringer Zahl')
else:
    print('\n✅ Alle Gewichte im Normbereich!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 5: MSS-Werte 2013-2019 interpolieren
# ═══════════════════════════════════════════════════════════════

def areal_interpolate(overlay_gdf, df_werte, jahr, value_cols):
    """Flaechengewichteter Mittelwert: alte -> neue Geometrie."""
    merged = overlay_gdf.merge(df_werte, on='plr_id_alt', how='left')
    results = []
    for plr_neu, group in merged.groupby('plr_id'):
        row = {'plr_id': plr_neu, 'year': jahr}
        gewicht_summe = group['gewicht'].sum()
        for col in value_cols:
            if gewicht_summe > 0:
                wert = (group[col] * group['gewicht']).sum() / gewicht_summe
            else:
                wert = np.nan
            row[col] = round(wert, 4) if pd.notna(wert) else np.nan
        results.append(row)
    return pd.DataFrame(results)


VALUE_COLS = ['arbeitslosigkeit', 'sgb2_quote', 'kinderarmut']

# ⚠️ Dateipfade anpassen an deine tatsaechlichen Dateinamen!
MSS_PATHS = {
    2013: '../data/raw/mss_2013.xlsx',
    2015: '../data/raw/mss_2015.xlsx',
    2017: '../data/raw/mss_2017.xlsx',
    2019: '../data/raw/mss_2019.xlsx',
}

interpolated_dfs = []

for jahr, path in MSS_PATHS.items():
    if not os.path.exists(path):
        print(f'⚠️ {jahr}: Datei nicht gefunden ({path}) - uebersprungen')
        continue

    df_jahr = load_mss_year(path, jahr)
    print(f'✅ {jahr} geladen: {len(df_jahr)} PLR (Rohdaten, alte Geometrie)')

    df_interp = areal_interpolate(overlay, df_jahr, jahr, VALUE_COLS)
    interpolated_dfs.append(df_interp)
    print(f'   -> interpoliert auf neue Geometrie: {len(df_interp)} PLR')

if interpolated_dfs:
    df_interpoliert = pd.concat(interpolated_dfs, ignore_index=True)
    print(f'\n✅ Gesamt interpoliert: {df_interpoliert.shape}')
else:
    df_interpoliert = pd.DataFrame(columns=['plr_id','year']+VALUE_COLS)
    print('\n⚠️ Keine Jahre interpoliert - Dateipfade pruefen!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 6: 2021/2023/2025 direkt laden (bereits neue Geometrie) + zusammenfuehren
# ═══════════════════════════════════════════════════════════════

MSS_PATHS_NEU = {
    2021: '../data/raw/mss_2021.xlsx',
    2023: '../data/raw/mss_2023.xlsx',
    2025: '../data/raw/mss_2025.xlsx',
}

direct_dfs = []

for jahr, path in MSS_PATHS_NEU.items():
    if not os.path.exists(path):
        print(f'⚠️ {jahr}: Datei nicht gefunden ({path}) - uebersprungen')
        continue

    df_jahr = load_mss_year(path, jahr)
    # load_mss_year() ist eine GEMEINSAME Funktion fuer alte UND neue
    # Jahre und benennt die ID-Spalte deshalb IMMER zu 'plr_id_alt'.
    # Fuer 2021/2023/2025 (bereits neue 542-PLR-Geometrie, keine
    # Interpolation noetig) muss sie hier explizit zu 'plr_id' umbenannt
    # werden, damit sie zur Zielspalte der interpolierten Jahre passt.
    df_jahr = df_jahr.rename(columns={'plr_id_alt': 'plr_id'})

    keep = ['plr_id', 'year'] + [c for c in VALUE_COLS if c in df_jahr.columns]
    direct_dfs.append(df_jahr[keep])
    print(f'✅ {jahr}: {len(df_jahr)} PLR (bereits neue Geometrie, direkt uebernommen)')

if direct_dfs:
    df_direkt = pd.concat(direct_dfs, ignore_index=True)
    df_mss = pd.concat([df_interpoliert, df_direkt], ignore_index=True)
else:
    df_mss = df_interpoliert.copy()

df_mss = df_mss.sort_values(['plr_id', 'year']).reset_index(drop=True)

print(f'\n✅ HARMONISIERTES GESAMTDATASET:')
print(f'   Shape: {df_mss.shape}')
print(f'   PLR:   {df_mss["plr_id"].nunique()}')
print(f'   Jahre: {sorted(df_mss["year"].unique())}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 7: Qualitaetskontrolle der Interpolation
# ═══════════════════════════════════════════════════════════════

# Vergleich: Original-Mittelwert (alte Geometrie) vs interpolierter Mittelwert (neue Geometrie)
for jahr, path in MSS_PATHS.items():
    if not os.path.exists(path):
        continue
    df_original = load_mss_year(path, jahr)
    if 'arbeitslosigkeit' not in df_original.columns:
        continue
    original_mean = df_original['arbeitslosigkeit'].mean()
    interp_subset = df_mss[df_mss['year']==jahr]
    if len(interp_subset) == 0:
        continue
    interp_mean = interp_subset['arbeitslosigkeit'].mean()
    diff = abs(original_mean - interp_mean)
    status = '✅' if diff < 0.5 else '⚠️'
    print(f'{status} {jahr}: Original={original_mean:.3f}%  '
          f'Interpoliert={interp_mean:.3f}%  Diff={diff:.3f}%-Pkt')

# Karte: Beispieljahr visualisieren (letztes verfuegbares interpoliertes Jahr)
verfuegbare_jahre_interp = sorted(df_mss[df_mss['year'].isin(MSS_PATHS.keys())]['year'].unique())
if verfuegbare_jahre_interp:
    beispiel_jahr = verfuegbare_jahre_interp[-1]
    gdf_check = gdf_neu.merge(
        df_mss[df_mss['year']==beispiel_jahr][['plr_id','arbeitslosigkeit']], on='plr_id'
    )
    fig, ax = plt.subplots(figsize=(9, 8))
    gdf_check.plot(column='arbeitslosigkeit', cmap='RdYlGn_r', legend=True, ax=ax,
                   edgecolor='white', linewidth=0.2)
    ax.set_title(f'Arbeitslosigkeit (S1) {beispiel_jahr}\n(interpoliert auf neue LOR-Geometrie)',
                 fontweight='bold')
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(f'../results/02_interpolation_check_{beispiel_jahr}.png', dpi=150)
    plt.show()
    print(f'✅ Qualitaetskontrolle abgeschlossen! (Karte: {beispiel_jahr})')
else:
    print('⚠️ Keine interpolierten Jahre fuer Kartenvergleich verfuegbar')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 8: Geometrische Zusatzfeatures (auf neuer Geometrie!)
# ═══════════════════════════════════════════════════════════════
# Hinweis: OSM-basierte Zusatzfeatures (OePNV, Gruenflaechen, Laerm)
# wurden bewusst entfernt, da die externe OpenStreetMap-Anbindung im
# Bearbeitungszeitraum nicht zuverlaessig funktionierte (instabile
# Verbindung, teilweise Fallback auf Zufallswerte). Diese Zelle
# berechnet nur noch die eigenstaendig aus echter Geometrie
# abgeleiteten raeumlichen Merkmale (schnell, keine externe Anbindung).

gdf_neu['centroid_x'] = gdf_neu.geometry.centroid.x
gdf_neu['centroid_y'] = gdf_neu.geometry.centroid.y
alex = Point(392456, 5820118)
gdf_neu['dist_zentrum'] = (gdf_neu.geometry.centroid.distance(alex) / 1000).round(2)

print(f'✅ Geometrische Merkmale berechnet fuer {len(gdf_neu)} PLR')
print(f'   dist_zentrum: {gdf_neu["dist_zentrum"].min():.1f} – '
      f'{gdf_neu["dist_zentrum"].max():.1f} km')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 9: Alles zusammenfuehren + kontinuierlichen Status-Index bilden
# ═══════════════════════════════════════════════════════════════
# WICHTIG: Die amtlichen MSS-Tabellen liefern S1/S3/S4 nur als Einzelwerte
# (Prozentanteile) sowie separat einen KATEGORIALEN Status-Index (4 Klassen,
# stufenweises Verfahren des Senats). Fuer die Regression mit LSTM/TFT wird
# ein eigener KONTINUIERLICHER Status-Index als z-Score-Mittel der drei
# konsistent verfuegbaren Indikatoren gebildet - vorzeichenkorrigiert, sodass
# ein HOEHERER Wert eine GUENSTIGERE soziale Lage bedeutet (weniger Arbeits-
# losigkeit, Transferbezug, Kinderarmut). Dies ist eine vereinfachte, selbst
# konstruierte Naeherung an den amtlichen Index, nicht mit diesem identisch
# (siehe Methodikkapitel 3.2 / Limitationen 6.3).

static_cols = ['plr_id', 'centroid_x', 'centroid_y', 'dist_zentrum']
# Hinweis: oepnv_stops/gruen_anteil/laerm_db bewusst NICHT hier,
# da FEATURE_COLS (Zelle 11) sie nicht mehr verwendet - siehe Begruendung dort.
df_static = pd.DataFrame(gdf_neu[static_cols].drop(columns=[]))

df_combined = df_mss.merge(df_static, on='plr_id', how='left')

# Eigener kontinuierlicher Status-Index (z-Score-Mittel, invertiert)
z_cols = ['arbeitslosigkeit', 'sgb2_quote', 'kinderarmut']
z_cols_present = [c for c in z_cols if c in df_combined.columns]

means = df_combined[z_cols_present].mean()
stds  = df_combined[z_cols_present].std()

z_scores = (df_combined[z_cols_present] - means) / stds
df_combined['status_index'] = -z_scores.mean(axis=1)  # invertiert: hoeher = guenstiger

print(f'✅ Kombiniertes Dataset: {df_combined.shape}')
print(f'   Spalten: {df_combined.columns.tolist()}')
print(f'\n📊 Eigener status_index (kontinuierlich):')
print(df_combined['status_index'].describe().round(3))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 10: Raeumlicher Lag, Ziel-Variable, Bereinigung
# ═══════════════════════════════════════════════════════════════

df_combined = df_combined.sort_values(['plr_id', 'year'])

# Fehlende Werte interpolieren
numeric_cols = df_combined.select_dtypes(include=[np.number]).columns
df_combined[numeric_cols] = (
    df_combined.groupby('plr_id')[numeric_cols]
    .apply(lambda x: x.interpolate(method='linear'))
    .reset_index(drop=True)
)
df_combined[numeric_cols] = df_combined[numeric_cols].fillna(
    df_combined[numeric_cols].median())

# Ausreisser (3-Sigma)
for col in ['status_index', 'arbeitslosigkeit', 'sgb2_quote']:
    if col not in df_combined.columns:
        continue
    mean, std = df_combined[col].mean(), df_combined[col].std()
    before = len(df_combined)
    df_combined = df_combined[abs(df_combined[col]-mean) <= 3*std]
    if before != len(df_combined):
        print(f'⚠️ {col}: {before-len(df_combined)} Ausreisser entfernt')

# Raeumlicher Lag (echte Nachbarschaft ueber Queen Contiguity)
try:
    from libpysal.weights import Queen
    w = Queen.from_dataframe(gdf_neu)
    w.transform = 'r'
    plr_list = gdf_neu['plr_id'].tolist()
    lag_rows = []
    for year in df_combined['year'].unique():
        df_yr = df_combined[df_combined['year']==year].set_index('plr_id')
        for i, plr_id in enumerate(plr_list):
            if plr_id not in df_yr.index:
                continue
            neigh = [plr_list[n] for n in w.neighbors.get(i, [])
                     if plr_list[n] in df_yr.index]
            lag = df_yr.loc[neigh, 'status_index'].mean() if neigh else df_yr.loc[plr_id,'status_index']
            lag_rows.append({'plr_id': plr_id, 'year': year, 'status_lag': round(float(lag),4)})
    df_lag = pd.DataFrame(lag_rows)
    df_combined = df_combined.merge(df_lag, on=['plr_id','year'], how='left')
    print('✅ Raeumlicher Lag (Queen-Nachbarschaft) berechnet')
except Exception as e:
    print(f'⚠️ Queen-Lag fehlgeschlagen ({e}) - Fallback: Jahresmittelwert')
    df_combined['status_lag'] = df_combined.groupby('year')['status_index'].transform('mean')

# Ziel-Variable
df_combined['status_next'] = df_combined.groupby('plr_id')['status_index'].shift(-1)
df_combined['delta_status'] = df_combined['status_next'] - df_combined['status_index']

# ── WICHTIG fuer 2027-Prognose: Snapshot VOR dem dropna sichern ────
# Das juengste Jahr (2025) hat kein status_next (kein Folgejahr bekannt)
# und wuerde durch dropna() komplett verloren gehen - inkl. seiner
# echten Feature-Werte, die man fuer eine 2027-Inferenz braucht!
juengstes_jahr = df_combined['year'].max()
df_letztes_jahr = df_combined[df_combined['year'] == juengstes_jahr].copy()
print(f'\n💾 {len(df_letztes_jahr)} PLR-Zeilen fuer Jahr {juengstes_jahr} '
      f'fuer spaetere Inferenz gesichert (Notebook 07)')

df_combined = df_combined.dropna(subset=['status_next', 'delta_status'])

year_map = {y: i for i, y in enumerate(sorted(df_combined['year'].unique()))}
df_combined['time_idx'] = df_combined['year'].map(year_map)

print(f'\n✅ Finale Bereinigung: {df_combined.shape}')
print(f'   Jahre: {sorted(df_combined["year"].unique())}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 11: FEATURE_COLS, TARGET festlegen
# ═══════════════════════════════════════════════════════════════
# Hinweis 1: 'langzeit_al' (S2) bewusst NICHT aufgenommen - Bedeutung
# wechselt zwischen Berichtsjahren (2013-17: Langzeit-AL, 2019+2021:
# fehlt, 2023+: Alleinerziehende) und bildet daher keine konsistente
# Zeitreihe (siehe Kapitel 3.2 / 6.3).
#
# Hinweis 2: 'oepnv_stops', 'gruen_anteil', 'laerm_db' bewusst NICHT
# aufgenommen. Diese OSM-basierten Zusatzfeatures konnten im
# Bearbeitungszeitraum nicht zuverlaessig ueber OpenStreetMap bezogen
# werden (instabile externe Anbindung, teilweise Fallback auf
# Zufallswerte).
#
# Hinweis 3: 'status_index' bewusst NICHT als Eingabe-Feature
# aufgenommen, obwohl es weiterhin im DataFrame vorhanden bleibt
# (wird fuer status_lag und status_next/Zielvariable benoetigt).
# Da status_index direkt aus S1/S3/S4 (z-Score-Mittel) berechnet
# wird, waere es als zusaetzliches Feature stark redundant zu den
# ohnehin enthaltenen Einzelindikatoren und wuerde unnoetige
# Kollinearitaet erzeugen. Die Modellierung beschraenkt sich daher
# auf die amtlich erhobenen Kernindikatoren (S1, S3, S4) sowie die
# eigenstaendig berechneten raeumlichen Merkmale (dist_zentrum,
# status_lag) - siehe Kapitel 3.2 / 6.3.

TARGET = 'status_next'

ALL_FEATURES = [
    'arbeitslosigkeit', 'sgb2_quote', 'kinderarmut',
    'dist_zentrum', 'status_lag',
]

FEATURE_COLS = [f for f in ALL_FEATURES
                if f in df_combined.columns and df_combined[f].notna().sum() > 0]

print(f'✅ Ziel-Variable: {TARGET}')
print(f'✅ Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'   (status_index bleibt im DataFrame erhalten, aber NICHT als Feature)')

not_used = [f for f in ALL_FEATURES if f not in FEATURE_COLS]
if not_used:
    print(f'⚠️ Nicht verfuegbar: {not_used}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 12: Split (zeitlich) & Normalisierung
# ═══════════════════════════════════════════════════════════════
# WICHTIG: Das juengste geladene Jahr (z.B. 2023) hat kein status_next
# und wird in Zelle 10 bereits als Zeile entfernt - es dient nur noch
# indirekt als Zielwert fuer das vorletzte Jahr. Die effektiv nutzbaren
# 'Anker-Jahre' sind also ein Jahr weniger als die Anzahl geladener
# MSS-Dateien. SEQ_LEN wird so gewaehlt, dass nach Abzug der noetigen
# Historie-Jahre mindestens 3 Anker-Jahre fuer Train/Val/Test uebrig
# bleiben - sonst waeren manche Splits leer (0 Sequenzen)!

alle_jahre = sorted(df_combined['year'].unique())
n = len(alle_jahre)
print(f'Verfuegbare Anker-Jahre: {alle_jahre} ({n} Jahre)')

if n < 4:
    raise RuntimeError(
        f'Nur {n} Anker-Jahre verfuegbar - mindestens 4 werden benoetigt '
        '(3 fuer Train/Val/Test + 1 Historie). Bitte MSS-Dateien pruefen.'
    )

# SEQ_LEN so waehlen, dass mindestens 3 Anker-Jahre uebrig bleiben
SEQ_LEN = max(1, n - 3)
eligible_jahre = alle_jahre[SEQ_LEN:]
print(f'SEQ_LEN (automatisch bestimmt): {SEQ_LEN}')
print(f'Nutzbare Anker-Jahre (genug Historie): {eligible_jahre}')

train_jahre = [eligible_jahre[0]]
val_jahre   = [eligible_jahre[1]]
test_jahre  = eligible_jahre[2:]  # Rest (meist 1 Jahr, kann mehr sein)

print(f'\nTrain: {train_jahre}  (Historie: {alle_jahre[:SEQ_LEN]})')
print(f'Val:   {val_jahre}')
print(f'Test:  {test_jahre}')

train = df_combined[df_combined['year'].isin(train_jahre)].copy()
val   = df_combined[df_combined['year'].isin(val_jahre)].copy()
test  = df_combined[df_combined['year'].isin(test_jahre)].copy()

scaler = StandardScaler()
train_scaled = train.copy()
val_scaled   = val.copy()
test_scaled  = test.copy()
df_full_scaled = df_combined.copy()

train_scaled[FEATURE_COLS] = scaler.fit_transform(train[FEATURE_COLS])
val_scaled[FEATURE_COLS]   = scaler.transform(val[FEATURE_COLS])
test_scaled[FEATURE_COLS]  = scaler.transform(test[FEATURE_COLS])
df_full_scaled[FEATURE_COLS] = scaler.transform(df_combined[FEATURE_COLS])

print(f'\n✅ Split: Train={len(train_scaled)} Val={len(val_scaled)} Test={len(test_scaled)}')
print(f'⚠️  SEQ_LEN={SEQ_LEN} wird in Zelle 13 mit in meta.pkl gespeichert '
      f'und von allen weiteren Notebooks automatisch uebernommen.')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ZELLE 13: Speichern (ECHTE Daten fuer Fine-Tuning)
# ═══════════════════════════════════════════════════════════════

train_scaled.to_csv('../data/final/train.csv', index=False)
val_scaled.to_csv('../data/final/val.csv', index=False)
test_scaled.to_csv('../data/final/test.csv', index=False)
df_full_scaled.to_csv('../data/final/df_full.csv', index=False)
df_combined.to_csv('../data/final/df_roh.csv', index=False)

# ── Fuer 2027-Prognose: juengstes Jahr (skaliert) separat speichern ──
df_letztes_jahr_scaled = df_letztes_jahr.copy()
df_letztes_jahr_scaled[FEATURE_COLS] = scaler.transform(df_letztes_jahr[FEATURE_COLS])
df_letztes_jahr_scaled.to_csv('../data/final/df_inferenz_2025.csv', index=False)
print(f'✅ Inferenz-Daten ({juengstes_jahr}) gespeichert: '
      f'{len(df_letztes_jahr_scaled)} PLR -> data/final/df_inferenz_2025.csv')

# Geometrie (echte Polygone) fuer Kartenerstellung in Notebook 06 speichern
gdf_neu[['plr_id', 'geometry']].to_file(
    '../data/final/plr_polygone.gpkg', driver='GPKG'
)
print('✅ Polygon-Geometrie gespeichert: data/final/plr_polygone.gpkg')

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)
with open('../models/target.pkl', 'wb') as f:
    pickle.dump(TARGET, f)

meta = {
    'n_plr': df_combined['plr_id'].nunique(),
    'jahre': alle_jahre, 'train_jahre': train_jahre,
    'val_jahre': val_jahre, 'test_jahre': test_jahre,
    'feature_cols': FEATURE_COLS, 'target': TARGET,
    'seq_len': SEQ_LEN,
}
with open('../models/meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('✅ ALLE ECHTEN DATEN GESPEICHERT!')
print('\n📁 data/final/  (echte, interpolierte Daten - fuer Fine-Tuning & SHAP)')
for fname in ['train.csv','val.csv','test.csv','df_full.csv','df_roh.csv']:
    p = f'../data/final/{fname}'
    print(f'   {fname:<15} {pd.read_csv(p).shape[0]:>4} Zeilen')

print(f'\n✅ Ziel: {TARGET} | Features: {len(FEATURE_COLS)}')
print('\n→ Weiter mit Notebook 03 (LSTM Pre-Training + Fine-Tuning)')